<a href="https://colab.research.google.com/github/gulshan0201/DL/blob/main/DL_lab5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

predict words by using auto encoder, I love ...... python code

In [1]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.utils import to_categorical

#Sample text data
text = "I love machine learning I love deep learning I love python"

#Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1

#Create input sequences
input_sequences = []
token_list = tokenizer.texts_to_sequences([text])[0]

for i in range(1, len(token_list)):
    n_gram_seq = token_list[:i+1]
    input_sequences.append(n_gram_seq)

#Padding
max_seq_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')

#Split into X and y
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

#One-hot encoding
y = to_categorical(y, num_classes=total_words)

#Model (Autoencoder-style LSTM)
input_layer = Input(shape=(max_seq_len - 1,))
embedding = Embedding(total_words, 10)(input_layer)

#Encoder
encoder = LSTM(64)(embedding)

# Decoder (predict next word)
output = Dense(total_words, activation='softmax')(encoder)

model = Model(input_layer, output)

#Compile & Train
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X, y, epochs=200, verbose=1)

#Function: Predict Top-K words with probabilities
def predict_next_words(model, tokenizer, text, max_seq_len, top_k=5):
    token_list = tokenizer.texts_to_sequences([text])[0]
    token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding='pre')

    predictions = model.predict(token_list, verbose=0)[0]

    #Get top K predictions
    top_indices = predictions.argsort()[-top_k:][::-1]

    results = []
    for idx in top_indices:
        for word, index in tokenizer.word_index.items():
            if index == idx:
                results.append((word, float(predictions[idx])))

    return results

#Test
seed_text = "I love"
results = predict_next_words(model, tokenizer, seed_text, max_seq_len, top_k=5)

print("\nNext word predictions with probabilities:")
for word, prob in results:
    print(f"{word} : {prob:.4f}")

Epoch 1/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.1000 - loss: 1.9461
Epoch 2/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.2000 - loss: 1.9426
Epoch 3/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.2000 - loss: 1.9390
Epoch 4/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.1000 - loss: 1.9354
Epoch 5/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.2000 - loss: 1.9316
Epoch 6/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.3000 - loss: 1.9277
Epoch 7/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.3000 - loss: 1.9235
Epoch 8/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.3000 - loss: 1.9189
Epoch 9/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.3000 - loss: 1.9139
Epoch 10/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.3000 - loss: 1.9083
Epoch 11/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 0.3000 - loss: 1.9022
Epoch 12/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.3000 - loss

In [2]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, GRU, SimpleRNN, Embedding, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ╡ Sample dataset
data = {
    "text": [
        "Machine learning is a field of artificial intelligence that uses statistical techniques",
        "Deep learning is a subset of machine learning that uses neural networks",
        "Natural language processing helps machines understand human language"
    ],
    "summary": [
        "ML uses statistics",
        "Deep learning uses neural networks",
        "NLP understands language"
    ]
}

df = pd.DataFrame(data)

# ╡ Tokenization
text_tokenizer = Tokenizer()
text_tokenizer.fit_on_texts(df['text'])

summary_tokenizer = Tokenizer()
summary_tokenizer.fit_on_texts(df['summary'])

text_vocab_size = len(text_tokenizer.word_index) + 1
summary_vocab_size = len(summary_tokenizer.word_index) + 1

# ╡ Convert to sequences
encoder_sequences = text_tokenizer.texts_to_sequences(df['text'])
decoder_sequences = summary_tokenizer.texts_to_sequences(df['summary'])

# ╡ Padding
max_text_len = max(len(seq) for seq in encoder_sequences)
max_summary_len = max(len(seq) for seq in decoder_sequences)

encoder_input = pad_sequences(encoder_sequences, maxlen=max_text_len, padding='post')
decoder_input = pad_sequences(decoder_sequences, maxlen=max_summary_len, padding='post')

# Shift decoder target
decoder_target = np.zeros_like(decoder_input)
decoder_target[:, :-1] = decoder_input[:, 1:]

# ╡ Choose model type: 'lstm' / 'gru' / 'rnn'
model_type = 'lstm'

latent_dim = 64

# ╡ Encoder (layer definition)
encoder_inputs = Input(shape=(max_text_len,))
enc_emb = Embedding(text_vocab_size, 64)(encoder_inputs)

encoder_states = []
encoder_outputs_training = None # Output of the encoder layer for the training model

if model_type == 'lstm':
    encoder = LSTM(latent_dim, return_state=True) # This 'encoder' is the layer instance
    encoder_outputs_training, state_h, state_c = encoder(enc_emb)
    encoder_states = [state_h, state_c]
elif model_type == 'gru':
    encoder = GRU(latent_dim, return_state=True) # This 'encoder' is the layer instance
    encoder_outputs_training, state_h = encoder(enc_emb)
    encoder_states = [state_h]
else: # SimpleRNN
    encoder = SimpleRNN(latent_dim, return_state=True) # This 'encoder' is the layer instance
    encoder_outputs_training, state_h = encoder(enc_emb)
    encoder_states = [state_h]

# ╡ Decoder (layer definition)
decoder_inputs = Input(shape=(max_summary_len,))
dec_emb = Embedding(summary_vocab_size, 64)(decoder_inputs)

decoder_outputs_training = None # Output of the decoder layer for the training model

if model_type == 'lstm':
    decoder = LSTM(latent_dim, return_sequences=True, return_state=True) # This 'decoder' is the layer instance
    decoder_outputs_training, _, _ = decoder(dec_emb, initial_state=encoder_states)
elif model_type == 'gru':
    decoder = GRU(latent_dim, return_sequences=True, return_state=True) # This 'decoder' is the layer instance
    decoder_outputs_training, _ = decoder(dec_emb, initial_state=encoder_states)
else: # SimpleRNN
    decoder = SimpleRNN(latent_dim, return_sequences=True, return_state=True) # This 'decoder' is the layer instance
    decoder_outputs_training, _ = decoder(dec_emb, initial_state=encoder_states)

# ╡ Output layer
decoder_dense = Dense(summary_vocab_size, activation='softmax')
decoder_outputs_training = decoder_dense(decoder_outputs_training)

# ╡ Training Model
model = Model([encoder_inputs, decoder_inputs], decoder_outputs_training)

# ╡ Compile
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

# ╡ Train
model.fit(
    [encoder_input, decoder_input],
    np.expand_dims(decoder_target, -1),
    epochs=200,
    verbose=1
)

# ╡ Inference Models
# Encoder inference model
encoder_model = Model(encoder_inputs, encoder_states)

# Decoder inference model
decoder_state_input_h = Input(shape=(latent_dim,), name='decoder_hidden_input')
decoder_states_inputs_inference = [decoder_state_input_h]
if model_type == 'lstm':
    decoder_state_input_c = Input(shape=(latent_dim,), name='decoder_cell_input')
    decoder_states_inputs_inference += [decoder_state_input_c]

    decoder_outputs_inference, state_h_output, state_c_output = decoder(
        dec_emb, initial_state=decoder_states_inputs_inference
    )
    decoder_states_output = [state_h_output, state_c_output]
elif model_type == 'gru':
    decoder_outputs_inference, state_h_output = decoder(
        dec_emb, initial_state=decoder_states_inputs_inference
    )
    decoder_states_output = [state_h_output]
else: # SimpleRNN
    decoder_outputs_inference, state_h_output = decoder(
        dec_emb, initial_state=decoder_states_inputs_inference
    )
    decoder_states_output = [state_h_output]

decoder_outputs_inference = decoder_dense(decoder_outputs_inference)
decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs_inference,
    [decoder_outputs_inference] + decoder_states_output
)

# Create a reverse word index for efficient lookup
reverse_summary_word_index = dict(map(reversed, summary_tokenizer.word_index.items()))

# ╡ Inference (Prediction)
def summarize(text):
    seq = text_tokenizer.texts_to_sequences([text])
    seq = pad_sequences(seq, maxlen=max_text_len, padding='post')

    # Get the initial states from the encoder
    states_value = encoder_model.predict(seq)

    # Initialize the target sequence with a start token.
    # If no explicit start token in summary_tokenizer, use 0 (padding).
    target_seq = np.zeros((1, 1)) # Start with padding token, which is ID 0

    output_sentence = []
    stop_condition = False

    while not stop_condition:
        if model_type == 'lstm':
            output_tokens, h, c = decoder_model.predict([target_seq] + states_value, verbose=0) # Add verbose=0 to avoid output flood
            states_value = [h, c]
        else: # gru or simple_rnn
            output_tokens, h = decoder_model.predict([target_seq] + states_value, verbose=0) # Add verbose=0
            states_value = [h]

        # Sample a token (greedy decoding)
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = reverse_summary_word_index.get(sampled_token_index)

        # Exit condition: either hit max length or find stop word (implicit stop for padding token 0)
        # Assuming 0 is a padding token and serves as an implicit stop.
        if sampled_word is None or sampled_token_index == 0 or len(output_sentence) >= max_summary_len:
            stop_condition = True
        else:
            output_sentence.append(sampled_word)

        # Update the target sequence (of length 1)
        target_seq = np.array([[sampled_token_index]])

    return " ".join(output_sentence)

# ╡ Test
print(summarize("Machine learning is a field of artificial intelligence"))

Epoch 1/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - loss: 2.4007
Epoch 2/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - loss: 2.3888
Epoch 3/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - loss: 2.3768
Epoch 4/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step - loss: 2.3645
Epoch 5/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - loss: 2.3516
Epoch 6/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step - loss: 2.3378
Epoch 7/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step - loss: 2.3228
Epoch 8/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step - loss: 2.3063
Epoch 9/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step - loss: 2.2880
Epoch 10/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - loss: 2.2675
Epoch 11/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - loss: 2.2442
Epoch 12/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - loss: 2.2175
Epoch 13/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - loss: 2.1868
Epoch 14/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step - loss: 2.1512
Epoch 15/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - loss: 2.1095
Epoch 16/200
1/